In [ ]:
# Importing libraries

import pandas as pd
import numpy as np
import re
from datetime import datetime
from google_play_scraper import app, reviews, Sort
from langdetect import detect
import os

In [ ]:
def is_english_text(text):
    """Return True when the text looks like English."""
    if pd.isna(text):
        return False

    text = str(text).strip()
    if not text:
        return False

    try:
        return detect(text) == "en"
    except Exception:
        return text.isascii()


def clean_text(text):
    """Standardize review text and return null for non-English text."""
    if not is_english_text(text):
        return None

    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

In [13]:
CBE_BANK_ID = 'com.combanketh.mobilebanking'

In [14]:
app_info = app(CBE_BANK_ID)
# app_info

In [15]:
print( "=" * 50 )
print( "App Information" )
print( "=" * 50 )

# printing some key information about the app
print(f"Title: {app_info['title']}")
print(f"Developer: {app_info['developer']}")
print(f"Rating: {app_info['ratings']}")
print(f"Number of Reviews: {app_info['reviews']}")
print(f"score: {app_info['score']}")
print(f"Installs: {app_info['installs']}")
print(f"reviews: {app_info['reviews']}")

App Information
Title: Commercial Bank of Ethiopia
Developer: Commercial Bank Of Ethiopia
Rating: 48323
Number of Reviews: 436
score: 4.157025
Installs: 5,000,000+
reviews: 436


In [19]:
result, continuation_token = reviews(CBE_BANK_ID, lang='en', country='us', sort=Sort.NEWEST, count=500)

In [21]:
len(result)

500

In [22]:
result[1]

{'reviewId': '06f6640c-b65c-43e4-88ef-0a79be8b9534',
 'userName': 'Abdi Lc',
 'userImage': 'https://play-lh.googleusercontent.com/a-/ALV-UjVVNd4xFs9wCfPd0n91qzJemrsPN5ldS_wGPSIRJtPISU5GgNbR',
 'content': "it's a good application",
 'score': 5,
 'thumbsUpCount': 0,
 'reviewCreatedVersion': '5.3.0',
 'at': datetime.datetime(2026, 5, 13, 20, 28, 58),
 'replyContent': None,
 'repliedAt': None,
 'appVersion': '5.3.0'}

Review Text: User feedback (e.g., "Love the UI, but it crashes often").
Rating: 1–5 stars.
Date: Posting date (YYYY-MM-DD).
Bank / App Name: e.g., "Commercial Bank of Ethiopia Mobile".
Source: Google Play.

In [23]:
raw_data = []

for review in result:
    raw_data.append(
        {
            "review_id": review['reviewId'],
            "rating": review['score'],
            "review": review['content'],
            "date": review['at'],
            "bank": "CBE",
            'source': 'Google Play Store'
        }
    )

In [45]:
df_raw = pd.DataFrame(raw_data)
df_raw.head()

,review_id,rating,review,date,bank,source
0,982b1262-3d54-4a12-811a-8b26b7ecc777,5,best app for financial sector,2026-05-14 05:44:01,CBE,Google Play Store
1,06f6640c-b65c-43e4-88ef-0a79be8b9534,5,it's a good application,2026-05-13 20:28:58,CBE,Google Play Store
2,eda74236-f7f3-4422-a139-a181e832bc27,5,thank you cbe,2026-05-13 17:16:37,CBE,Google Play Store
3,ff53332b-2e76-46d6-83d3-f93f968a4b18,5,is good,2026-05-13 16:18:45,CBE,Google Play Store
4,363a5616-ed3d-4274-85ee-77071067f81d,5,wow,2026-05-13 12:19:17,CBE,Google Play Store


In [39]:
df_raw.dtypes

review_id               str
rating                int64
review                  str
date         datetime64[us]
bank                    str
source                  str
dtype: object

In [29]:
# what's the count for each rating?
df_raw['rating'].value_counts()


rating
5    339
1     73
4     41
3     35
2     12
Name: count, dtype: int64

In [49]:
df_raw['review'].apply(clean_text)

0      best app for financial sector
1            it's a good application
2                      thank you cbe
3                            is good
4                                wow
                   ...              
495                       sestem esy
496               account not raning
497                              NaN
498                              bad
499                              nic
Name: review, Length: 500, dtype: str

Let's audit our three common data quality problems:
1. Missing values
2. Duplicate reviews  
3. Inconsistent date formats

In [50]:
# 1. Missing Values Check

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

  review_id      : OK
  rating         : OK
  review         : OK
  date           : OK
  bank           : OK
  source         : OK


In [51]:
# 2 . Duplicate Check

duplicate_count = df_raw.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count} ({(duplicate_count / len(df_raw) * 100).round(2)}%)")

Duplicate Rows: 0 (0.0%)


In [52]:
duplicate_rows_by_id = df_raw.duplicated(['review_id'], keep=False).sum()
print(f"Duplicate Rows (by review_id): {duplicate_rows_by_id} ({(duplicate_rows_by_id / len(df_raw) * 100).round(2)}%)")

Duplicate Rows (by review_id): 0 (0.0%)


In [53]:
# 3. Date Format Check
# Check if 'date' column is in datetime format:  YYYY-MM-DD

date_point = df_raw['date'].iloc[0]
date_point

'2026-05-14'

In [54]:
df_raw['date'].dtype

<StringDtype(storage='python', na_value=nan)>

---

Now we fix each problem, one at a time.  
We work on a **copy** so we can always compare back to the raw data.

### Strategy Overview

| Problem | Strategy | Tool |
|---------|----------|------|
| Missing critical data | Drop the row | `df.dropna()` |
| Duplicate reviews | Keep first occurrence | `df.drop_duplicates()` |
| Inconsistent dates | Parse and reformat | `pd.to_datetime()` |
| Messy text | Strip whitespace / remove junk | `str.strip()`, `re.sub()` |

In [41]:
df_raw.dropna(inplace=True)

In [42]:
df_raw.drop_duplicates(inplace=True)

In [46]:
print("Before cleaning:")
print(df_raw['date'].iloc[0])
# Convert 'date' column to datetime format
df_raw['date'] = pd.to_datetime(df_raw['date']).dt.strftime('%Y-%m-%d')
print("After cleaning:")
print(df_raw['date'].iloc[0])


Before cleaning:
2026-05-14 05:44:01
After cleaning:
2026-05-14


Clean review text

In [ ]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

In [55]:
# validate rating 1 <= rating <= 5

df_raw = df_raw[(df_raw['rating'] >= 1) & (df_raw['rating'] <= 5)]
df_raw['rating'] = df_raw['rating'].astype(int)

In [57]:
df_clean = df_raw.drop(columns=['review_id'])

In [58]:
df_clean.sort_values(by='date', ascending=False, inplace=True)
df_clean.reset_index(drop=True, inplace=True)

In [59]:
df_clean.shape

(500, 5)

In [65]:
# Save to CSV
import os
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/cbe_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/cbe_bank_reviews_clean.csv


Cleaning pipeline documentation

In [63]:
# Document cleaning steps: compare "before" (raw_data) and "after" (df_clean).
# Explanations are printed inline for transparency.

df_before = pd.DataFrame(raw_data)        # reconstructed original from raw_data
df_after = df_clean.copy()                # cleaned dataframe produced earlier

# Summaries
print("Summary:")
print(f"  Rows before : {len(df_before)}")
print(f"  Rows after  : {len(df_after)}")
print()

# Missing values
print("Missing values (before -> after):")
miss_b = df_before.isnull().sum()
miss_a = df_after.isnull().sum()
for col in df_before.columns:
    after_val = miss_a[col] if col in miss_a.index else "dropped"
    print(f"  {col:<12}: {miss_b[col]:>3} -> {after_val}")
print("Rationale: dropped rows with critical missing values to ensure required fields are present.")
print()

# Duplicates
dups_b = df_before.duplicated().sum()
dups_a = df_after.duplicated().sum()
dups_by_id_b = df_before.duplicated(['review_id'], keep=False).sum()
dups_by_id_a = df_after.duplicated(['review_id'], keep=False).sum() if 'review_id' in df_after.columns else 0
print("Duplicates:")
print(f"  Any-rows duplicated (before -> after): {dups_b} -> {dups_a}")
print(f"  Duplicated by review_id (before -> after): {dups_by_id_b} -> {dups_by_id_a}")
print("Rationale: removed exact duplicates and duplicates by review_id to avoid double-counting feedback.")
print()

# Rating validation
invalid_ratings_b = df_before[~df_before['rating'].between(1,5)].shape[0]
invalid_ratings_a = df_after[~df_after['rating'].between(1,5)].shape[0]
print("Rating validation:")
print(f"  Invalid ratings (before -> after): {invalid_ratings_b} -> {invalid_ratings_a}")
print("Rationale: enforced ratings between 1 and 5 and cast to int for consistency.")
print()

# Date formats/types
print("Date column type (before -> after):")
print(f"  before: {df_before['date'].dtype}   ->   after: {df_after['date'].dtype}")
print("Rationale: normalized dates to YYYY-MM-DD strings for easy downstream export/consumption.")
print()

# Text cleaning examples
print("Review text examples (raw -> cleaned preview):")
sample_n = 5
raw_preview = df_before['review'].head(sample_n).reset_index(drop=True)
cleaned_preview_from_before = raw_preview.apply(clean_text)
after_preview = df_after['review'].head(sample_n).reset_index(drop=True)
preview_df = pd.DataFrame({"raw": raw_preview, "cleaned_from_raw": cleaned_preview_from_before, "after_df": after_preview})
print(preview_df.to_string(index=False))
print("Rationale: collapsed whitespace and stripped edges; removed non-English or empty reviews earlier to keep meaningful text.")
print()

# File output
print(f"Final cleaned CSV saved at: {output_path}")
print("Overall rationale: drop rows with missing critical fields, deduplicate, normalize types and text to produce a reliable, reproducible dataset.")



Summary:
  Rows before : 500
  Rows after  : 500

Missing values (before -> after):
  review_id   :   0 -> dropped
  rating      :   0 -> 0
  review      :   0 -> 0
  date        :   0 -> 0
  bank        :   0 -> 0
  source      :   0 -> 0
Rationale: dropped rows with critical missing values to ensure required fields are present.

Duplicates:
  Any-rows duplicated (before -> after): 0 -> 19
  Duplicated by review_id (before -> after): 0 -> 0
Rationale: removed exact duplicates and duplicates by review_id to avoid double-counting feedback.

Rating validation:
  Invalid ratings (before -> after): 0 -> 0
Rationale: enforced ratings between 1 and 5 and cast to int for consistency.

Date column type (before -> after):
  before: datetime64[us]   ->   after: str
Rationale: normalized dates to YYYY-MM-DD strings for easy downstream export/consumption.

Review text examples (raw -> cleaned preview):
                          raw              cleaned_from_raw                      after_df
best a